# Edgeflow Diagnostics Report

This notebook loads `tools/edgeflow_verbose_diagnostics.jsonl` (newline-delimited JSON),
produces a compact tabular summary of rings that look problematic, and generates
annotated plots (pre/post) for quick inspection.

Usage: run the cells sequentially. If you prefer to run the helper script, use:
`python tools/plot_edgeflow_diagnostics.py --jsonl tools/edgeflow_verbose_diagnostics.jsonl --outdir tmp/plots`

In [ ]:
# Imports and helpers
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def load_jsonl(path: Path):
    entries = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            try:
                entries.append(json.loads(line))
            except Exception:
                continue
    return entries


def canonical_rows_from_entries(entries):
    rows = []
    for ent in entries:
        for r in ent.get("rows") or []:
            # attach top-level metadata useful for summary
            row = dict(r)
            row["_timestamp"] = ent.get("timestamp")
            row["_stage"] = ent.get("stage")
            rows.append(row)
    return rows

In [ ]:
# Path to JSONL (change if your file is elsewhere)
jsonl_path = Path("tools") / "edgeflow_verbose_diagnostics.jsonl"
assert jsonl_path.exists(), f"Expected diagnostics JSONL at {jsonl_path}"
entries = load_jsonl(jsonl_path)
rows = canonical_rows_from_entries(entries)
len(rows)

In [ ]:
# Build a pandas summary table highlighting potentially-problematic rings
def safe_get(d, k, default=None):
    try:
        return d.get(k, default)
    except Exception:
        return default


summary_rows = []
for r in rows:
    zi = int(safe_get(r, "zi", -1))
    min_final_raw = safe_get(r, "min_final_raw")
    violations = safe_get(r, "enforcement_violations_count") or 0
    gt = safe_get(r, "Gt_stats")
    gz = safe_get(r, "Gz_stats")
    dom = safe_get(r, "dom_counts")
    lifted = r.get("lift_delta") is not None and any(
        [v > 1e-9 for v in (r.get("lift_delta") or [])],
    )
    summary_rows.append(
        {
            "zi": zi,
            "timestamp": r.get("_timestamp"),
            "min_final_raw": min_final_raw,
            "violations": violations,
            "lifted_any": lifted,
            "Gt_mean": (gt.get("mean") if gt else None),
            "Gz_mean": (gz.get("mean") if gz else None),
            "dom_vert": (dom.get("vert") if dom else None),
            "dom_diap": (dom.get("diap") if dom else None),
            "dom_diam": (dom.get("diam") if dom else None),
        },
    )

df = pd.DataFrame(summary_rows)
# sort to show problematic rings first: violations desc, lifted_any, min_final_raw asc
df_sorted = df.sort_values(
    by=["violations", "lifted_any", "min_final_raw"], ascending=[False, False, True],
)
df_sorted.head(40)

In [ ]:
# Plot the top N problematic rings (by violations or lifted flag)
def plot_row_inline(r, title=None):
    thetas = r.get("theta_sample")
    if thetas is None and r.get("R_raw_sample") is not None:
        T = len(r.get("R_raw_sample"))
        thetas = np.arange(T) * (2.0 * np.pi / float(T))
    else:
        thetas = np.asarray(thetas, dtype=float)
    r_raw = np.asarray(r.get("R_raw_sample") or [], dtype=float)
    r_new_raw = np.asarray(r.get("R_new_raw_sample") or [], dtype=float)
    env_post = np.asarray(
        r.get("Env_to_use_raw_post") or r.get("Env_to_use_sample") or [], dtype=float,
    )
    lift_delta = np.asarray(r.get("lift_delta") or np.zeros_like(r_raw), dtype=float)
    violations = r.get("enforcement_violations_indices") or []
    plt.figure(figsize=(10, 4))
    if title is None:
        title = f"zi={r.get('zi')} ts={r.get('_timestamp')}"
    plt.title(title)
    plt.plot(thetas, r_raw, label="R_raw", linestyle="--")
    plt.plot(thetas, r_new_raw, label="R_new_raw", linestyle="-")
    if env_post.size:
        plt.plot(thetas, env_post, label="Env_post", linestyle=":")
    mask = lift_delta > 1e-9
    if np.any(mask):
        plt.scatter(thetas[mask], r_new_raw[mask], color="orange", label="lifted")
    if violations:
        vi = np.asarray(violations, dtype=int)
        plt.scatter(
            thetas[vi], r_new_raw[vi], color="red", marker="x", s=60, label="violations",
        )
    plt.legend()
    plt.xlabel("theta (rad)")
    plt.ylabel("radius")
    plt.grid(True)
    plt.show()


# choose top entries to plot
to_plot = df_sorted.head(6)["zi"].tolist()
for zi in to_plot:
    # find a matching row (choose first occurrence)
    match = next((r for r in rows if int(r.get("zi", -1)) == int(zi)), None)
    if match is not None:
        plot_row_inline(match)